Set up a very simple system with a single particle and shifted Lj wall forces. Then explicitly change the position of the particle along the x-axis (y=z=0) and compute the potential energy. Finally, plot this potential energy as a function of x.
'
what we need is basically a custom external force in every direction which is mostly 0. And only when you are within some distance to the "wall" at +- L/2 you switch on a Lennard Jones interaction
 All that we want is an external potential that is zero for -L/2 + cutoff < x < L/2 - cutoff and at this cutoff value the potential should start growing like LJ after the minimum

## OpenMM

In [122]:
import ase.io
import gsd.hoomd
import numpy.typing as npt
import openmm
from openmm import app
from openmm import unit
from openmm import CustomExternalForce


def read_xyz_file(filename: str) -> (list[str], npt.NDArray[float], npt.NDArray[float]):
    if not filename.endswith(".xyz"):
        raise ValueError("The file must have the .xyz extension.")
    atoms = ase.io.read(filename, format="extxyz")
    cell = atoms.get_cell()[:]
    assert cell.shape == (3, 3)
    return atoms.get_chemical_symbols(), atoms.get_positions(), cell

In [123]:
types, positions, cell = read_xyz_file('test_system.xyz')

In [124]:
topology = app.topology.Topology()
chain = topology.addChain()
residue = topology.addResidue("res1", chain)

for t in types:
    topology.addAtom(t, None, residue)

#topology.setPeriodicBoxVectors(cell)

system = openmm.System()

temperature=298
timestep=0.005
collision_rate=1

integrator = openmm.LangevinIntegrator(temperature,collision_rate, timestep)

masses=[1,1]
box_length=1000

for t in types:
    system.addParticle(masses[0])
    
platform_name= 'Reference'
    
platform = openmm.Platform.getPlatformByName(platform_name)
        
simulation = app.Simulation(topology, system, integrator, platform)

simulation.context.setPositions(positions)

slj_potential = CustomExternalForce(
    "step(-box_length/2 + r_cut + delta, box_length/2 - r_cut - delta) * "
    "4*epsilon*((radius/(x-delta))^12+(radius/(x-delta))^6) +(radius/(y-delta))^12+(radius/(y-delta))^6)(radius/(z-delta))^12+(radius/(z-delta))^6)"
    "r_cut = radius * 2 **(1/6)"
    "delta = radius - 1"
    )

slj_potential.addGlobalParameter("epsilon", 1.0)

slj_potential.addGlobalParameter("box_length",
                                    box_length) #.value_in_unit(self._nanometer))

slj_potential.addPerParticleParameter("x")
slj_potential.addPerParticleParameter("y")
slj_potential.addPerParticleParameter("z")
slj_potential.addPerParticleParameter("radius")

system.addForce(slj_potential)

        
simulation.reporters.append(app.StateDataReporter(None, 1, time=True,
                                                      kineticEnergy=False, potentialEnergy=True, temperature=False,
                                                      speed=False, append=None))



for i in range(0,101):
    
    positions[0][0]+=10
    print(positions)

    simulation.step(1)

[[ 9.35000002 -0.64999998 -0.64999998]]
#"Time (ps)","Potential Energy (kJ/mole)"
0.005,0.0
[[19.35000002 -0.64999998 -0.64999998]]
0.01,0.0
[[29.35000002 -0.64999998 -0.64999998]]
0.015,0.0
[[39.35000002 -0.64999998 -0.64999998]]
0.02,0.0
[[49.35000002 -0.64999998 -0.64999998]]
0.025,0.0
[[59.35000002 -0.64999998 -0.64999998]]
0.030000000000000002,0.0
[[69.35000002 -0.64999998 -0.64999998]]
0.035,0.0
[[79.35000002 -0.64999998 -0.64999998]]
0.04,0.0
[[89.35000002 -0.64999998 -0.64999998]]
0.045,0.0
[[99.35000002 -0.64999998 -0.64999998]]
0.049999999999999996,0.0
[[109.35000002  -0.64999998  -0.64999998]]
0.05499999999999999,0.0
[[119.35000002  -0.64999998  -0.64999998]]
0.05999999999999999,0.0
[[129.35000002  -0.64999998  -0.64999998]]
0.06499999999999999,0.0
[[139.35000002  -0.64999998  -0.64999998]]
0.06999999999999999,0.0
[[149.35000002  -0.64999998  -0.64999998]]
0.075,0.0
[[159.35000002  -0.64999998  -0.64999998]]
0.08,0.0
[[169.35000002  -0.64999998  -0.64999998]]
0.085,0.0
[[179

## HOOMD

In [125]:
import hoomd
import gsd.hoomd 

from hoomd import md

import numpy as np
import math
import itertools

In [126]:
N_particles = 1

box_length = 1000 #length of box

spacing = 1.3
K = math.ceil(N_particles ** (1 / 3))
L = K * spacing
x = np.linspace(-L / 2, L / 2, K, endpoint=False)
position = list(itertools.product(x, repeat=3))

radiusN=105
radiusP=85

frame = gsd.hoomd.Frame()
frame.particles.N = N_particles
frame.particles.position = position[0:N_particles]
frame.particles.typeid = [0]* N_particles
frame.configuration.box = [box_length, box_length, box_length, 0, 0, 0]

frame.particles.types = ['N']

In [127]:
with gsd.hoomd.open(name='test_system.gsd', mode='w') as f:
    f.append(frame)

In [129]:
hoomd.context.initialize("--mode=cpu")
system=hoomd.init.read_gsd('test_system.gsd',frame=-1)
snapshot = system.take_snapshot()

notice(2): Group "all" created containing 1 particles


In [130]:
upper_wall_x = hoomd.md.wall.plane(origin=(L/2,0,0),normal=(-1,0,0),inside=True)
lower_wall_x = hoomd.md.wall.plane(origin=(-L/2,0,0),normal=(1,0,0),inside=True)
upper_wall_y = hoomd.md.wall.plane(origin=(0,L/2,0),normal=(0,-1,0),inside=True)
lower_wall_y = hoomd.md.wall.plane(origin=(0,-L/2,0),normal=(0,1,0),inside=True)
upper_wall_z = hoomd.md.wall.plane(origin=(0,0,L/2),normal=(0,0,-1),inside=True)
lower_wall_z = hoomd.md.wall.plane(origin=(0,0,-L/2),normal=(0,0,1),inside=True)

wall_group = hoomd.md.wall.group(upper_wall_x,lower_wall_x,upper_wall_y,lower_wall_y,upper_wall_z,lower_wall_z)
wall_force = hoomd.md.wall.slj(wall_group,r_cut=radiusN*(2.0**(1.0/6.0)))
wall_force.force_coeff.set('N',epsilon=1,sigma=radiusN,alpha=0,r_cut=radiusN*(2.0**(1.0/6.0)))

notice(2): Notice: slj set d_max=1.0


In [131]:
hoomd.md.integrate.mode_standard(dt=0.005)
gamma=1
all = hoomd.group.all()

log = hoomd.analyze.log(filename=None,
                      quantities=['potential_energy', 'temperature'],
                      period=1,
                      overwrite=True,
                      phase=-1)


seed=1
integrator = hoomd.md.integrate.nvt(group=all,kT=1, tau=1/gamma)
integrator.randomize_velocities(seed=seed)

In [132]:
print(snapshot.box)

snapshot.box.periodic = np.array([True,True,True])

Box: Lx=1000.0 Ly=1000.0 Lz=1000.0 xy=0.0 xz=0.0 yz=0.0 dimensions=3


In [133]:
hoomd.md.integrate.mode_standard(dt=0.005)
gamma=1
all = hoomd.group.all()

log = hoomd.analyze.log(filename=None,
                      quantities=['potential_energy', 'temperature'],
                      period=1,
                      overwrite=True,
                      phase=-1)


seed=1
integrator = hoomd.md.integrate.nvt(group=all,kT=1, tau=1/gamma)
integrator.randomize_velocities(seed=seed)

*Warning*: compute.thermo already specified for a group with name __nvt_all


In [100]:


for i in range(0,101):
    
    
    hoomd.run(1)
    
    
    snapshot.particles.position[0][0] +=10
    print(snapshot.particles.position)
    
    print(log.query("potential_energy"))

*Warning*: compute.thermo: given a group with 0 degrees of freedom.
            overriding ndof=1 to avoid divide by 0 errors
*Warning*: compute.thermo: given a group with 0 degrees of freedom.
            overriding ndof=1 to avoid divide by 0 errors


** starting run **
Time 00:00:21 | Step 1 / 1 | TPS 4.09356 | ETA 00:00:00
Average TPS: 4.06365
---------
** run complete **


*Warning*: compute.thermo: given a group with 0 degrees of freedom.
            overriding ndof=1 to avoid divide by 0 errors
*Warning*: compute.thermo: given a group with 0 degrees of freedom.
            overriding ndof=1 to avoid divide by 0 errors


[[ 9.35 -0.65 -0.65]]
0.0
** starting run **
Time 00:00:21 | Step 2 / 2 | TPS 100000 | ETA 00:00:00
Average TPS: 3952.57
---------
** run complete **


*Warning*: compute.thermo: given a group with 0 degrees of freedom.
            overriding ndof=1 to avoid divide by 0 errors
*Warning*: compute.thermo: given a group with 0 degrees of freedom.
            overriding ndof=1 to avoid divide by 0 errors


...

** starting run **
Time 00:00:22 | Step 100 / 100 | TPS 58823.5 | ETA 00:00:00
Average TPS: 5025.13
---------
** run complete **


*Warning*: compute.thermo: given a group with 0 degrees of freedom.


[[ 9.9935e+02 -6.5000e-01 -6.5000e-01]]
0.0


            overriding ndof=1 to avoid divide by 0 errors
*Warning*: compute.thermo: given a group with 0 degrees of freedom.
            overriding ndof=1 to avoid divide by 0 errors


** starting run **
Time 00:00:22 | Step 101 / 101 | TPS 71428.6 | ETA 00:00:00
Average TPS: 4926.11
---------
** run complete **
[[ 1.00935e+03 -6.50000e-01 -6.50000e-01]]
0.0


In [101]:
frame = gsd.hoomd.Frame()
frame.particles.N = N_particles
frame.particles.position = snapshot.particles.position
frame.particles.typeid = [0]* N_particles
frame.configuration.box = [box_length, box_length, box_length, 0, 0, 0]

with gsd.hoomd.open(name='move.gsd', mode='w') as f:
    f.append(frame)

In [102]:
r_cut=radiusN*(2.0**(1.0/6.0))

r_cut

117.85851507248417

In [88]:
snapshot.particles.position

array([[ 9.8935e+02, -6.5000e-01, -6.5000e-01]], dtype=float32)